# Solar Energy Optimizer - Interactive Monitoring Dashboard

## Notebook overview

This notebook receives solar-energy telemetry from an Arduino Uno through Proteus and a virtual serial-port pair, processes each sample, logs readings, and presents them in an interactive Matplotlib dashboard. It can also replay a previously recorded CSV file.

## Workflow

1. Load plotting, data-processing, and serial libraries.
2. Define the common reader interface and live serial reader.
3. Define the CSV playback reader.
4. Process readings, calculate power and status, and log live data.
5. Build the interactive plots, status banner, and dashboard controls.
6. Choose live serial or CSV replay, then launch the dashboard.

## Requirements

- Python packages: matplotlib, numpy, pandas, and pyserial for live serial input.
- For live input: Arduino Uno firmware, Proteus, and Virtual Serial Port Driver (or an equivalent virtual null-modem tool).
- Default serial configuration: Proteus on COM1 and notebook on COM2, both at 9600 baud.

## Run instructions

1. Install the Python packages in the Jupyter environment.
2. Compile the Arduino sketch for an Uno and configure its sensor pins (A0, A1, and A2).
3. Create the virtual COM-port pair before starting the Proteus simulation.
4. Configure Proteus COMPIM for the Proteus-side port and 9600 baud, then run the simulation.
5. Run notebook cells from top to bottom. In the final cell, use SerialSensorReader for live input or CSVSensorReader to replay sensor_log.csv.
6. Use the dashboard status banner and charts; adjust the temperature slider, pause/resume updates, or reset the in-memory buffer as needed.

## Outputs

- An interactive dashboard with irradiance, temperature/voltage, temperature distribution, power, and current-status displays.
- Live samples are written to sensor_log.csv with timestamp, sensor values, and inferred status. Existing four-column logs are upgraded with inferred status labels.

### Stage 1: Load libraries and serial support

**Purpose:** Import plotting, data-processing, and standard-library modules, select the interactive Matplotlib backend, and make PySerial available for live input.

**Details:** See the adjacent Python cell for the implementation.

In [ ]:
import matplotlib
matplotlib.use("TkAgg")  # Use TkAgg backend for matplotlib

import os
import tempfile
import time
import warnings
from abc import ABC, abstractmethod

# Ignore all user warnings
warnings.filterwarnings("ignore")

import matplotlib.animation as animation
import matplotlib.pyplot as plt
from matplotlib.widgets import Button, Slider
import numpy as np
import pandas as pd

# Safe serial import
try:
  import serial
except ImportError:
  serial = None

print("[Stage 1/6] Imports and dependencies loaded.")


### Stage 2: Define the reader interface and serial input

**Purpose:** Create the shared BaseSensorReader interface and SerialSensorReader, which reads and parses comma-separated irradiance, temperature, and voltage samples at 9600 baud.

**Details:** See the adjacent Python cell for the implementation.

In [ ]:
class BaseSensorReader(ABC):
  """Abstract Base Class defining the sensor telemetry interface."""

  @abstractmethod
  def connect(self) -> bool:
    """Establish connection to telemetry source."""
    pass

  @abstractmethod
  def read_sample(self) -> dict:
    """Read a single telemetry frame. Returns dict or None."""
    pass

  @abstractmethod
  def close(self):
    """Release communication and system resources."""
    pass


class SerialSensorReader(BaseSensorReader):
  """Strict driver for reading real-time telemetry from Arduino / Proteus COMPIM
  on COM2.

  Never generates synthetic data. Automatically waits/retries until COM2 is
  active.
  """

  def __init__(
      self,
      port: str = "COM2",
      baud_rate: int = 9600,
      timeout: float = 1.0,
      retry_forever: bool = True,
  ):
    self.port = port
    self.baud_rate = baud_rate
    self.timeout = timeout
    self.retry_forever = retry_forever
    self.ser = None

  def connect(self) -> bool:
    """Connects to COM2 with auto-retry loop for Proteus simulation."""
    if serial is None:
      print("[Error] 'pyserial' is not installed. Run: pip install pyserial")
      return False

    while True:
      try:
        self.ser = serial.Serial(
            self.port, self.baud_rate, timeout=self.timeout
        )
        time.sleep(1.0)
        print(
            f"[Serial Driver] Successfully connected to {self.port} at"
            f" {self.baud_rate} baud."
        )
        return True
      except Exception:
        if not self.retry_forever:
          print(f"[Serial Driver] Unable to open {self.port}.")
          return False
        print(
            f"[Serial Driver] Waiting for Proteus on {self.port}... (Retrying"
            " in 2s)"
        )
        time.sleep(2.0)

  def read_sample(self) -> dict:
    """Parse CSV telemetry from Proteus: 'irradiance,temperature,voltage'."""
    if not self.ser or not self.ser.is_open:
      return None
    try:
      line = self.ser.readline().decode("utf-8", errors="ignore").strip()
      if not line:
        return None
      parts = [float(p.strip()) for p in line.split(",")]
      if len(parts) >= 3:
        return {
            "timestamp": time.strftime("%H:%M:%S"),
            "irradiance": parts[0],
            "temperature": parts[1],
            "voltage": parts[2],
        }
    except (ValueError, IndexError, Exception):
      return None
    return None

  def close(self):
    """Safely closes the serial connection."""
    if self.ser and self.ser.is_open:
      self.ser.close()
      print("[Serial Driver] Port closed.")

print("[Stage 2/6] Sensor reader classes are ready.")


### Stage 3: Define CSV playback

**Purpose:** Create CSVSensorReader to replay records from sensor_log.csv in order, looping back to the first record after the last.

**Details:** See the adjacent Python cell for the implementation.

In [ ]:
class CSVSensorReader(BaseSensorReader):
  """Read recorded CSV telemetry without generating synthetic data."""

  def __init__(self, filepath: str = "sensor_log.csv"):
    self.filepath = filepath
    self.df = None
    self.cursor = 0

  def connect(self) -> bool:
    """Loads existing real telemetry records from CSV."""
    try:
      if not os.path.exists(self.filepath):
        print(f"[CSV Driver] '{self.filepath}' not found.")
        return False

      self.df = pd.read_csv(self.filepath)
      if len(self.df) == 0:
        print(
            f"[CSV Driver] '{self.filepath}' is empty. Awaiting real serial"
            " data."
        )
        return False

      self.cursor = 0
      print(f"[CSV Driver] Loaded {len(self.df)} logged records.")
      return True
    except Exception as e:
      print(f"[CSV Driver Error] Failed to read {self.filepath}: {e}")
      return False

  def read_sample(self) -> dict:
    if self.df is None or len(self.df) == 0:
      return None
    try:
      row = self.df.iloc[self.cursor]
      self.cursor = (self.cursor + 1) % len(self.df)
      return {
          "timestamp": str(row["timestamp"]),
          "irradiance": float(row["irradiance"]),
          "temperature": float(row["temperature"]),
          "voltage": float(row["voltage"]),
      }
    except Exception:
      return None

  def close(self):
    self.df = None

print("[Stage 3/6] CSV sensor reader is ready.")


### Stage 4: Process and log measurements

**Purpose:** Maintain the rolling sample buffer, calculate estimated power and status, compute summary statistics, and append live samples to the status-aware CSV log. Legacy four-column logs are upgraded while preserving existing rows.

**Details:** See the adjacent Python cell for the implementation.

In [ ]:
class DataProcessor:
  """Manages sliding telemetry buffer, real-time CSV disk logging, and
  vectorized statistical calculations using Pandas and NumPy.
  """

  def __init__(
      self,
      buffer_size: int = 50,
      log_file: str = "sensor_log.csv",
      auto_log_to_csv: bool = True,
  ):
    self.buffer_size = buffer_size
    self.log_file = log_file
    self.auto_log = auto_log_to_csv
    self.buffer = pd.DataFrame(
        columns=["timestamp", "irradiance", "temperature", "voltage", "status", "power"]
    )
    self._initialize_csv_header()

  @staticmethod
  def _classify_status(irr: float, temp: float, volt: float) -> str:
    """Assign an operating state using the dashboard alert thresholds."""
    if temp >= 65.0 or volt < 10.5:
      return "CRITICAL_ALERT"
    if irr < 40.0 or temp > 48.0:
      return "SUB_OPTIMAL"
    return "OPTIMAL"

  def _initialize_csv_header(self):
    """Create the status-and-source-aware CSV schema or upgrade legacy schemas."""
    expected_columns = [
        "timestamp", "irradiance", "temperature", "voltage", "status", "source"
    ]
    legacy_4_columns = expected_columns[:4]
    legacy_5_columns = expected_columns[:5]

    if not os.path.exists(self.log_file) or os.stat(self.log_file).st_size == 0:
      pd.DataFrame(columns=expected_columns).to_csv(self.log_file, index=False)
      return

    existing = pd.read_csv(self.log_file)
    if list(existing.columns) == legacy_4_columns:
      existing["status"] = existing.apply(
          lambda row: self._classify_status(
              float(row["irradiance"]),
              float(row["temperature"]),
              float(row["voltage"]),
          ),
          axis=1,
      )
      existing["source"] = "REAL"
      existing.to_csv(self.log_file, index=False)
      print("[CSV] Upgraded 4-column legacy log; added status and source='REAL'.")
    elif list(existing.columns) == legacy_5_columns:
      existing["source"] = "REAL"
      existing.to_csv(self.log_file, index=False)
      print("[CSV] Upgraded 5-column legacy log; added source='REAL'.")
    elif "source" not in existing.columns:
      existing["source"] = "REAL"
      existing.to_csv(self.log_file, index=False)

  def ingest(self, sample: dict, is_live_stream: bool = True):
    """Appends new real sample to rolling Pandas buffer and logs to CSV on disk."""
    if sample is None:
      return

    irr = float(sample["irradiance"])
    temp = float(sample["temperature"])
    volt = float(sample["voltage"])
    status = self._classify_status(irr, temp, volt)
    sample["status"] = status

    # Derived feature: estimated electrical power with thermal degradation
    power = (irr / 100.0) * volt * 6.5 * (1.0 - (temp - 25.0) * 0.004)
    sample["power"] = max(0.0, power)

    # 1. Update in-memory rolling Pandas buffer
    new_row = pd.DataFrame([sample])
    self.buffer = pd.concat([self.buffer, new_row], ignore_index=True)
    if len(self.buffer) > self.buffer_size:
      self.buffer = self.buffer.iloc[-self.buffer_size :].reset_index(drop=True)

    # 2. Append real telemetry to disk CSV log with source='REAL'
    if self.auto_log and is_live_stream:
      try:
        with open(self.log_file, "a", newline="") as f:
          f.write(
              f"{sample['timestamp']},{irr:.2f},{temp:.2f},{volt:.2f},{status},REAL\n"
          )
      except Exception as e:
        print(f"[Log Error] Failed to write to CSV: {e}")

  def clear(self):
    """Clears the in-memory buffer."""
    self.buffer = self.buffer.iloc[0:0]

  def compute_statistics(self) -> dict:
    """Performs vectorized numerical calculations on active buffer using NumPy."""
    if len(self.buffer) < 2:
      return {
          "mean_irr": 0.0,
          "mean_temp": 0.0,
          "std_temp": 0.0,
          "mean_volt": 0.0,
          "mean_power": 0.0,
          "peak_power": 0.0,
      }

    arr_irr = self.buffer["irradiance"].to_numpy(dtype=float)
    arr_temp = self.buffer["temperature"].to_numpy(dtype=float)
    arr_volt = self.buffer["voltage"].to_numpy(dtype=float)
    arr_power = self.buffer["power"].to_numpy(dtype=float)

    return {
        "mean_irr": float(np.mean(arr_irr)),
        "mean_temp": float(np.mean(arr_temp)),
        "std_temp": float(np.std(arr_temp)),
        "mean_volt": float(np.mean(arr_volt)),
        "mean_power": float(np.mean(arr_power)),
        "peak_power": float(np.max(arr_power)),
    }

print("[Stage 4/6] Data processor is ready.")

### Stage 5: Build the interactive dashboard

**Purpose:** Create the color-coded current-status banner, four telemetry plots, temperature-threshold slider, pause/resume button, and buffer-reset button.

**Details:** See the adjacent Python cell for the implementation.

In [ ]:
class InteractiveMonitoringDashboard:
  """Interactive Real-Time Monitoring GUI using Matplotlib animation & UI Widgets."""

  def __init__(self, reader: BaseSensorReader, processor: DataProcessor):
    self.reader = reader
    self.processor = processor
    self.is_paused = False
    self.temp_threshold = 65.0
    self.ani = None

    self.fig, self.axes = plt.subplots(2, 2, figsize=(13, 8.5))
    try:
      self.fig.canvas.manager.set_window_title(
          "Solar Energy Optimizer — Interactive Dashboard"
      )
    except Exception:
      pass
    self.fig.suptitle(
        "Solar Energy Optimizer — Interactive Monitoring Dashboard",
        fontsize=14,
        fontweight="bold",
    )
    self.status_display = self.fig.text(
        0.5,
        0.91,
        "Current Status: WAITING FOR DATA",
        ha="center",
        va="center",
        fontsize=12,
        fontweight="bold",
        color="#475569",
        bbox={"boxstyle": "round,pad=0.35", "facecolor": "#e2e8f0", "edgecolor": "none"},
    )
    plt.subplots_adjust(top=0.84, bottom=0.14, hspace=0.35, wspace=0.25)

    self.ax_line_irr = self.axes[0, 0]  # Panel 1: Line Chart (Irradiance)
    self.ax_line_temp = self.axes[
        0, 1
    ]  # Panel 2: Line Chart (Temperature & Voltage)
    self.ax_hist = self.axes[
        1, 0
    ]  # Panel 3: Histogram (Temperature Distribution)
    self.ax_scatter = self.axes[1, 1]  # Panel 4: Irradiance vs. power

    # 1. Slider: Alert Temperature Threshold
    ax_slider = plt.axes([0.18, 0.04, 0.35, 0.03])
    self.slider_thresh = Slider(
        ax=ax_slider,
        label="Alert Temp (°C) ",
        valmin=45.0,
        valmax=85.0,
        valinit=self.temp_threshold,
        valstep=1.0,
        color="#ef4444",
    )
    self.slider_thresh.on_changed(self._on_slider_change)

    # 2. Button: Pause / Resume Stream
    ax_btn_pause = plt.axes([0.62, 0.03, 0.15, 0.045])
    self.btn_pause = Button(
        ax_btn_pause, "Pause Stream", color="#e2e8f0", hovercolor="#cbd5e1"
    )
    self.btn_pause.on_clicked(self._toggle_pause)

    # 3. Button: Reset Active Buffer
    ax_btn_reset = plt.axes([0.79, 0.03, 0.15, 0.045])
    self.btn_reset = Button(
        ax_btn_reset, "Reset Buffer", color="#fee2e2", hovercolor="#fca5a5"
    )
    self.btn_reset.on_clicked(self._reset_buffer)

  def _on_slider_change(self, val):
    self.temp_threshold = float(val)

  def _toggle_pause(self, event):
    self.is_paused = not self.is_paused
    if self.is_paused:
      self.btn_pause.label.set_text("Resume Stream")
      self.btn_pause.color = "#bbf7d0"
    else:
      self.btn_pause.label.set_text("Pause Stream")
      self.btn_pause.color = "#e2e8f0"
    self.fig.canvas.draw_idle()

  def _reset_buffer(self, event):
    self.processor.clear()
    self.fig.canvas.draw_idle()

  def update_frame(self, frame):
    """Animation tick callback to read data, update buffer, and redraw plots."""
    if not self.is_paused:
      try:
        sample = self.reader.read_sample()
      except KeyboardInterrupt:
        print("[System] Terminating dashboard...")
        plt.close(self.fig)
        return
      if sample is not None:
        is_live = isinstance(self.reader, SerialSensorReader)
        self.processor.ingest(sample, is_live_stream=is_live)

    df = self.processor.buffer
    if len(df) == 0:
      status = "WAITING FOR DATA"
    else:
      status = str(df.iloc[-1].get("status", "UNKNOWN"))

    status_styles = {
        "OPTIMAL": ("#166534", "#dcfce7"),
        "SUB_OPTIMAL": ("#92400e", "#fef3c7"),
        "CRITICAL_ALERT": ("#991b1b", "#fee2e2"),
    }
    text_color, background_color = status_styles.get(
        status, ("#475569", "#e2e8f0")
    )
    self.status_display.set_text(f"Current Status: {status}")
    self.status_display.set_color(text_color)
    self.status_display.get_bbox_patch().set_facecolor(background_color)

    if len(df) < 2:
      return

    stats = self.processor.compute_statistics()

    # 1. Line Chart: Solar Irradiance
    self.ax_line_irr.clear()
    self.ax_line_irr.plot(
        df.index,
        df["irradiance"],
        color="#f59e0b",
        lw=2,
        marker="o",
        markersize=3,
        label="Irradiance (%)",
    )
    self.ax_line_irr.axhline(
        40.0,
        color="orange",
        linestyle="--",
        alpha=0.7,
        label="Shading Alert (40%)",
    )
    self.ax_line_irr.set_ylim(0, 105)
    self.ax_line_irr.set_title(
        f"Solar Irradiance Trend | Avg: {stats['mean_irr']:.1f}%",
        fontsize=10,
        fontweight="bold",
    )
    self.ax_line_irr.set_ylabel("Irradiance (%)")
    self.ax_line_irr.legend(loc="lower left", fontsize=8)
    self.ax_line_irr.grid(True, linestyle=":", alpha=0.6)

    # 2. Line Chart: Thermal & Battery Dynamics
    self.ax_line_temp.clear()
    self.ax_line_temp.plot(
        df.index,
        df["temperature"],
        color="#ef4444",
        lw=2,
        label="Temperature (°C)",
    )
    self.ax_line_temp.plot(
        df.index,
        df["voltage"] * 5.0,
        color="#0284c7",
        lw=2,
        label="Voltage (Scaled x5 V)",
    )
    self.ax_line_temp.axhline(
        self.temp_threshold,
        color="darkred",
        linestyle="--",
        lw=1.5,
        label=f"Alert Threshold ({self.temp_threshold:.0f}°C)",
    )
    self.ax_line_temp.set_title(
        f"Thermal & Storage Dynamics | Temp: {stats['mean_temp']:.1f}°C",
        fontsize=10,
        fontweight="bold",
    )
    self.ax_line_temp.set_ylabel("Values")
    self.ax_line_temp.legend(loc="upper left", fontsize=8)
    self.ax_line_temp.grid(True, linestyle=":", alpha=0.6)

    # 3. Histogram: Temperature Frequency Distribution
    self.ax_hist.clear()
    temps = df["temperature"].to_numpy(dtype=float)
    self.ax_hist.hist(
        temps, bins=8, color="#f87171", edgecolor="black", alpha=0.75
    )
    self.ax_hist.set_title(
        f"Temperature Histogram | Std Dev: {stats['std_temp']:.2f}°C",
        fontsize=10,
        fontweight="bold",
    )
    self.ax_hist.set_xlabel("Temperature (°C)")
    self.ax_hist.set_ylabel("Frequency")
    self.ax_hist.grid(True, linestyle=":", alpha=0.6)

    # 4. Scatter Plot: Irradiance vs. Generated Power
    self.ax_scatter.clear()
    self.ax_scatter.scatter(
        df["irradiance"],
        df["power"],
        c=df["temperature"],
        cmap="viridis",
        edgecolors="k",
        alpha=0.85,
    )
    self.ax_scatter.set_title(
        f"Scatter: Irradiance vs. Power | Mean P: {stats['mean_power']:.1f}W",
        fontsize=10,
        fontweight="bold",
    )
    self.ax_scatter.set_xlabel("Irradiance (%)")
    self.ax_scatter.set_ylabel("Power (Watts)")
    self.ax_scatter.grid(True, linestyle=":", alpha=0.6)

  def run(self):
    """Launches real-time Matplotlib animation loop."""
    self.ani = animation.FuncAnimation(
        self.fig, self.update_frame, interval=250, cache_frame_data=False
    )
    plt.show(block=True)

print("[Stage 5/6] Interactive dashboard class is ready.")


### Stage 6: Choose an input source and launch

**Purpose:** Choose SerialSensorReader for live readings from Arduino/Proteus, or CSVSensorReader for recorded readings from sensor_log.csv; then launch the dashboard and release the reader when it closes.

**Details:** See the adjacent Python cell for the implementation.

In [ ]:
if __name__ == "__main__":
  # Retry the COM2 connection until the Proteus simulation is running.
  reader = SerialSensorReader(port="COM2", baud_rate=9600, retry_forever=True)
  reader.connect()

  # To replay the saved CSV instead, replace the two serial lines above with:
  # reader = CSVSensorReader(filepath="sensor_log.csv")
  # reader.connect()

  processor = DataProcessor(buffer_size=40)
  dashboard = InteractiveMonitoringDashboard(
      reader=reader, processor=processor
  )

  try:
    dashboard.run()
  except KeyboardInterrupt:
    print("[System] Terminating dashboard...")
  finally:
    reader.close()

print("[Stage 6/6] Dashboard stopped and serial port released.")
